In [3]:
import polars as pl
import tarfile
import os

# === NASTAVENÍ ===
original_dataset = "../protbert/datasets/dataset_merged.parquet"           # Původní data
RESULTS_PATTERN = "results/*.tsv"  # Cesta k výsledkům z clusteru
OUTPUT_CSV = "dataset_with_families.parquet"

df_original = pl.read_parquet(original_dataset)

# drop columns besides original_seq_full and reverse
df_original = df_original.select(["original_seq_full", "reverse"])

df_map = (
    df_original
    .select(pl.col("original_seq_full"))
    .unique()
    .sort("original_seq_full") # <--- Důležité pro reprodukovatelnost
    .filter(pl.col("original_seq_full").str.len_chars() >= 5)
    .with_row_index(name="id_numeric")
)

# 2. Zpracování výsledků
print("2. Agreguji CATH rodiny (Dominantní + Všechny)...")

ipro_cols = ["seq_id", "md5", "len", "analysis", "family_id", "desc", "start", "end", "score", "status", "date", "ipr_acc", "ipr_desc"]

df_cath_results = (
    pl.scan_csv(RESULTS_PATTERN, separator="\t", has_header=False, new_columns=ipro_cols, truncate_ragged_lines=True)
    # Filtrujeme jen CATH (Gene3D)
    .filter(pl.col("analysis") == "Gene3D")

    .with_columns([
        pl.col("seq_id").str.strip_prefix("seq_").cast(pl.UInt32).alias("id_numeric"),
        (pl.col("end") - pl.col("start")).alias("domain_length")
    ])

    # SEŘAZENÍ:
    # 1. Podle ID (aby se grupy zpracovávaly stejně)
    # 2. Podle délky domény sestupně (aby první řádek byl ten dominantní)
    .sort(["id_numeric", "domain_length"], descending=[False, True])

    # AGREGACE
    .group_by("id_numeric")
    .agg([
        # A) Dominantní: Vezme první hodnotu (díky sortu výše je to ta nejdelší)
        pl.col("family_id").first().alias("cath_dominant"),

        # B) Seznam všech: Unikátní hodnoty, SEŘAZENÉ (aby byl seznam vždy stejný), spojené středníkem
        pl.col("family_id").unique().sort().str.join(";").alias("cath_all")
    ])
    .collect()
)

# 3. Spojení všeho dohromady
print("3. Spojuji výsledky s původním datasetem...")

# Mapa + CATH data
df_mapping_final = df_map.join(df_cath_results, on="id_numeric", how="left")

# Původní data + Mapa
final_df = df_original.join(
    df_mapping_final.select(["original_seq_full", "cath_dominant", "cath_all"]),
    on="original_seq_full",
    how="left"
)

# 5. Statistiky pokrytí
total = final_df.height
found = final_df.filter(pl.col("cath_dominant").is_not_null()).height
missing = total - found
multi_domain = final_df.filter(pl.col("cath_all").str.contains(";")).height


# Statistika
missing = final_df.filter(pl.col("reverse") == False).filter(pl.col("cath_dominant").is_null()).height
missing_reversed = final_df.filter(pl.col("reverse") == True).filter(pl.col("cath_dominant").is_null()).height
print(f"\nStatistika:")
print(f"Celkem řádků: {final_df.height}")
print(f"Řádků bez nalezené rodiny: {missing} ({(missing/final_df.height)*100:.1f} %)")
print(f"Řádků bez nalezené rodiny(reverse): {missing_reversed} ({(missing_reversed/final_df.height)*100:.1f} %)")

print(f"Celkové pokrytí: {total - missing} ({((total-missing)/total)*100:.1f} %)")
print(f"Stále chybí:     {missing}")

final_df.select(["original_seq_full", "cath_dominant", "cath_all"]).write_parquet(OUTPUT_CSV)
final_df

2. Agreguji CATH rodiny (Dominantní + Všechny)...
3. Spojuji výsledky s původním datasetem...

Statistika:
Celkem řádků: 1949832
Řádků bez nalezené rodiny: 110883 (5.7 %)
Řádků bez nalezené rodiny(reverse): 100088 (5.1 %)
Celkové pokrytí: 1838949 (94.3 %)
Stále chybí:     110883


original_seq_full,reverse,cath_dominant,cath_all
str,bool,str,str
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…",false,"""G3DSA:1.20.58.60""","""G3DSA:1.10.238.10;G3DSA:1.20.5…"
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…",false,"""G3DSA:1.20.1270.60""","""G3DSA:1.10.555.10;G3DSA:1.20.1…"
"""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…",false,"""G3DSA:3.40.50.300""","""G3DSA:1.10.8.60;G3DSA:2.40.40.…"
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…",false,"""G3DSA:1.25.40.10""","""G3DSA:1.25.40.10;G3DSA:2.30.30…"
"""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…",false,"""G3DSA:2.170.270.10""","""G3DSA:2.170.270.10;G3DSA:3.30.…"
…,…,…,…
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""G3DSA:2.30.30.40""","""G3DSA:2.30.30.40"""
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""G3DSA:2.30.30.40""","""G3DSA:2.30.30.40"""
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""G3DSA:2.30.30.40""","""G3DSA:2.30.30.40"""


In [4]:
final_df.group_by("cath_dominant").count().sort("count", descending=True)

/tmp/ipykernel_90514/853346493.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  final_df.group_by("cath_dominant").count().sort("count", descending=True)


cath_dominant,count
str,u32
null,210971
"""G3DSA:2.30.30.40""",161171
"""G3DSA:2.40.50.140""",84376
"""G3DSA:2.30.42.10""",75082
"""G3DSA:1.10.10.60""",68965
…,…
"""G3DSA:3.30.1360.100""",2
"""G3DSA:1.20.81.10""",1
"""G3DSA:3.90.810.10""",1


In [ ]:
filtered = final_df.filter(pl.col("cath_dominant").is_null()).filter(pl.col("reverse") == False)

# print first 10 in fasta format

for i,r in enumerate(filtered.to_series().to_list()[:10]):
    print(f">Seq{i}\n{r}")
